In [1]:
# Calculate global mortality - requires ~75GB memory

In [2]:
import os
import xarray as xr
import numpy as np
import warnings
from utils.utils import get_scenario_config
from utils.mortality_utils import att_frac

In [3]:
# === Path config ===
MASKS_DIR = "/glade/work/awells/air_quality/BMR/masks/country/"
POP_DIR = "/glade/work/awells/air_quality/SSP_pop/SSP2/"
BMR_DIR = "/glade/derecho/scratch/awells/air_quality/BMR/"

In [4]:
# Load country masks
mask_file = "GBD_Country_Masks_0.10.nc"
mask_path = os.path.join(MASKS_DIR, mask_file)
masks = xr.open_dataarray(mask_path)

# Load population file
pop_file = "ssp2_coarse_grid_annual_2000-2100.nc"
pop_path = os.path.join(POP_DIR, pop_file)
population = xr.open_dataarray(pop_path)
pop = population.reindex_like(masks, method="nearest", tolerance=1e-9)

In [5]:
# === Calculate the scalar distributions (assuming normal dist.) ===
n_samples = 1000

# TMREL from GBD21
tmrel_mean = 32.4
tmrel_std = (35.7 - 29.1) / (2 * 1.96)
tmrel_samples = np.random.normal(tmrel_mean, tmrel_std, size=n_samples)

tmrel_da = xr.DataArray(
    tmrel_samples,
    dims=['samples'],
    coords={'samples': np.arange(n_samples)}
).astype("float32")

# Beta from RR per 10ppb
RR_10 = 1.074
RR_10_lower = 1.014
RR_10_upper = 1.137
beta_mean = np.log(RR_10) / 10
beta_std = (np.log(RR_10_upper) - np.log(RR_10_lower)) / (2 * 1.96 * 10)
beta_samples = np.random.normal(beta_mean, beta_std, size=n_samples)

beta_da = xr.DataArray(
    beta_samples,
    dims=['samples'],
    coords={'samples': np.arange(n_samples)}
).astype("float32")

# Load BMR for each country
bmr_file = f"GBD_BMR_Country_Mask_COPD_{n_samples}_samples_1990-2009.nc"
bmr_path = os.path.join(BMR_DIR, bmr_file)
BMR = xr.open_dataarray(bmr_path)  # three quantiles

In [ ]:
warnings.filterwarnings('ignore')

# === Scenario and path config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
model = "CESM2"
scenario = "SSP245_G6"

config = get_scenario_config(model, scenario)
ensemble_members = [3] # config["ensemble_members"][1:]
years = config["years"]

O3_DIR = f"/glade/work/awells/air_quality/{model}/ozone/OSDMA8_BC/"
SAVE_DIR = f"/glade/work/awells/air_quality/{model}/mortality/ozone/global/"

for ens_num in ensemble_members:
    print(f"Processing ensemble member {ens_num:02d}")
    dates = f"{years.start}-{years.stop - 1}"

    # Load ozone data
    o3_file = f"OSDMA8_BC_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
    o3_path = os.path.join(O3_DIR, o3_file)
    o3 = xr.open_dataarray(o3_path).astype("float32")
    o3 = o3.reindex_like(masks, method="nearest", tolerance=1e-9, fill_value=0)

    del o3_file, o3_path

    # make o3 dask-backed
    o3 = o3.chunk({'lat': 180, 'lon': 360})

    for year in years[:-1]:  # the last year doesn't exist in OSDMA8
        print(f"Processing year {year}")
        o3_year = o3.sel(year=year)
        AF = att_frac(o3_year, tmrel_da, beta_da).chunk({"samples": 10})

        POP = pop.sel(year=year).chunk({"lat": 180, "lon": 360})
        M = AF * BMR * POP
        global_M = M.sum(dim=("lat", "lon"))

        # mean over samples
        M_mean = global_M.mean(dim="samples")
        # median over samples
        M_median = global_M.quantile(0.5, dim="samples")
        # percentiles for 95% CI
        M_lower = global_M.quantile(0.025, dim="samples")
        M_upper = global_M.quantile(0.975, dim="samples")

        M_summary = xr.Dataset({
            "M_mean": M_mean.astype("float32").drop_vars("year"),
            "M_median": M_median.astype("float32").drop_vars("quantile"),
            "M_lower95": M_lower.astype("float32").drop_vars("quantile"),
            "M_upper95": M_upper.astype("float32").drop_vars("quantile")
        })

        del M_mean, M_median, M_lower, M_upper

        out_file = f"Global_mortality_stats_{model}_{scenario}_{ens_num:02d}_{year}.nc"
        out_path = os.path.join(SAVE_DIR, out_file)
        print(f"Saving to {out_path}")
        M_summary.to_netcdf(out_path)

        del o3_year, AF, POP, M, global_M, M_summary

    del o3

Processing ensemble member 03
Processing year 2020
Saving to /glade/work/awells/air_quality/CESM2/mortality/ozone/global/Global_mortality_stats_CESM2_SSP245_G6_03_2020.nc
Processing year 2021
Saving to /glade/work/awells/air_quality/CESM2/mortality/ozone/global/Global_mortality_stats_CESM2_SSP245_G6_03_2021.nc
Processing year 2022
Saving to /glade/work/awells/air_quality/CESM2/mortality/ozone/global/Global_mortality_stats_CESM2_SSP245_G6_03_2022.nc
Processing year 2023
Saving to /glade/work/awells/air_quality/CESM2/mortality/ozone/global/Global_mortality_stats_CESM2_SSP245_G6_03_2023.nc
Processing year 2024
Saving to /glade/work/awells/air_quality/CESM2/mortality/ozone/global/Global_mortality_stats_CESM2_SSP245_G6_03_2024.nc
Processing year 2025
Saving to /glade/work/awells/air_quality/CESM2/mortality/ozone/global/Global_mortality_stats_CESM2_SSP245_G6_03_2025.nc
Processing year 2026
Saving to /glade/work/awells/air_quality/CESM2/mortality/ozone/global/Global_mortality_stats_CESM2_SSP2